# JED trace-guided attack search

This notebook packages the attached framework dataset into `/kaggle/working` and starts the official competition server. It uses no external network or API.

In [ ]:
import glob
import os
import sys
import shutil
from pathlib import Path

sys.argv = [sys.argv[0]]
# First official run: bounded public-fixture/scorer baseline.
# Change to hybrid only after recording this baseline result.
os.environ.setdefault('JED_MODE', 'baseline')
os.environ.setdefault('JED_CANARY_COUNT', '24')

# Competition SDK input.
for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    dataset_root = str(Path(candidate).parent)
    if dataset_root not in sys.path:
        sys.path.insert(0, dataset_root)
    break
else:
    raise RuntimeError('Competition kaggle_evaluation input was not found')

# Framework dataset input. Set FRAMEWORK_ROOT only when discovery is ambiguous.
override = os.environ.get('FRAMEWORK_ROOT', '').strip()
if override:
    framework_root = Path(override)
else:
    matches = [Path(path).parent.parent for path in glob.glob(
        '/kaggle/input/**/jedfw/entrypoint.py', recursive=True
    )]
    # A private Dataset may contain the release zip instead of files.
    if not matches:
        archives = [Path(path) for path in glob.glob(
            '/kaggle/input/**/jed_attack_framework.zip', recursive=True
        )]
        if len(archives) == 1:
            extracted = Path('/kaggle/working/framework_dataset')
            shutil.unpack_archive(archives[0], extracted)
            matches = [Path(path).parent.parent for path in glob.glob(
                str(extracted / '**/jedfw/entrypoint.py'), recursive=True
            )]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one framework dataset, found: {matches}')
    framework_root = matches[0]

if not (framework_root / 'bundle_submission.py').is_file():
    raise RuntimeError(f'Invalid framework root: {framework_root}')
sys.path.insert(0, str(framework_root))
from bundle_submission import bundle

attack_path = bundle(framework_root, '/kaggle/working')
print(f'Bundled attack: {attack_path}')


In [ ]:
# Kaggle needs this file for a visible Save & Run. The official gateway
# writes the real score during the competition rerun.
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    Path('/kaggle/working/submission.csv').write_text(
        'Id,Score\n'
        'gpt_oss_public,0.0\n'
        'gpt_oss_private,0.0\n'
        'gemma_public,0.0\n'
        'gemma_private,0.0\n',
        encoding='utf-8',
    )
    print('submission.csv placeholder written')


In [ ]:
import kaggle_evaluation.jed_attack_134815.jed_attack_inference_server as server
server.JEDAttackInferenceServer().serve()
